# 03 · RAG: contract-aware invoice checking

**The finance question:** *"Is this invoice consistent with what we agreed with this supplier?"*

The pipeline has separate, inspectable stages: **extract → clean → chunk → embed → store**, then
**filter → vector + keyword search → Reciprocal Rank Fusion → citations**. The model *maps* invoice lines to
contract rates; **code** verifies the quote and compares the prices.

Real Gemini embeddings (`gemini-embedding-001`, 768 dims) and the real chat model. This notebook only **reads**
the demo database (documents ingested with `uv run python -m trustagent.rag.ingest`).

In [1]:
import datetime as dt
import math

import pandas as pd

from trustagent.config import get_settings
from trustagent.db.session import get_sessionmaker
from trustagent.rag.embeddings import GeminiEmbedder
from trustagent.rag.ingest import chunk_document, extract_pages, load_manifest, strip_boilerplate
from trustagent.rag.retrieve import rrf_merge, search

settings = get_settings()
pd.set_option("display.max_colwidth", 100)
session = get_sessionmaker()()
embedder = GeminiEmbedder()
print(
    f"Embeddings: {embedder.model}, {embedder.dim} dims | chunk size {settings.chunk_size}, "
    f"overlap {settings.chunk_overlap} | RRF k={settings.rrf_k}"
)

Embeddings: gemini-embedding-001, 768 dims | chunk size 800, overlap 120 | RRF k=60


## 1. The documents and their metadata

Metadata comes from a manifest, **never** from the model: which supplier, which contract, and the dates it is in
force. One Metro Cleaning contract is **expired**: its older, lower rates must never be used.

In [2]:
manifest = load_manifest()
pd.DataFrame(
    [
        {
            "document": m.document_id,
            "type": m.doc_type,
            "supplier": m.supplier_id,
            "from": m.effective_from,
            "to": m.effective_to,
            "title": m.title,
        }
        for m in manifest
    ]
)

,document,type,supplier,from,to,title
0,ABC-2026-017,contract,SUP-001,2026-02-15,2027-02-14,Supply Agreement - Office Consumables and Furniture
1,MC-2025-004,contract,SUP-002,2025-04-01,2027-03-31,Cleaning Services Agreement
2,MC-2024-001,contract,SUP-002,2024-04-01,2025-03-31,Cleaning Services Agreement (previous term)
3,DPC-2025-009,contract,SUP-003,2025-11-10,2026-11-09,Print Services Rate Card Agreement
4,FIN-POL-01,policy,NaN,2026-06-01,None,Procurement and Supplier Payments Policy Manual


## 2. Extract and clean

PDFs come with running headers, footers and page numbers. Left in, they'd appear in every chunk and pollute
keyword search. Lines repeated on most pages are removed.

In [3]:
policy = next(m for m in manifest if m.doc_type == "policy")
raw = extract_pages(settings.data_dir / policy.file)
clean = strip_boilerplate(raw)
print(f"{len(raw)} pages.  Page 4 raw, first lines:")
print("  " + "\n  ".join(raw[3].lines[:4]))
print("\nAfter cleaning:")
print("  " + "\n  ".join(clean[3].lines[:3]))

12 pages.  Page 4 raw, first lines:
  TrustCorp Holdings Ltd - FIN-POL-01 Procurement and Supplier Payments - v3.2
  Page 4
  Confidential - TrustCorp Holdings Ltd
  7 Transaction Pattern Monitoring (POL-003)

After cleaning:
  7 Transaction Pattern Monitoring (POL-003)
  7.1 An invoice for more than three times the supplier's historical average payment must be flagged for review
  before payment. Where a supplier has an agreed expected spending range, invoices above the range are


## 3. Structure-aware chunking

Split on **section headings first**, then pack whole clauses up to ~800 characters with ~120 characters of
overlap. A clause is never cut mid-sentence, and a rate table stays in one chunk. Every chunk starts with a
**context header** ("document | section"), which helps both the embedding and keyword search.

In [4]:
chunks = chunk_document(clean, policy)
print(len(chunks), "chunks from the policy manual")
pd.DataFrame([{"#": c.index, "page": c.page, "section": c.section, "chars": len(c.text)} for c in chunks]).head(14)

33 chunks from the policy manual


,#,page,section,chars
0,0,1,Preamble,216
1,1,1,1 Purpose and Scope,608
2,2,1,1 Purpose and Scope,487
3,3,1,2 Definitions,570
4,4,2,3 Roles and Segregation of Duties,595
5,5,2,4 Supplier Onboarding and Master Data,695
6,6,2,4 Supplier Onboarding and Master Data,375
7,7,3,5 Supplier Bank Account Changes (POL-001),287
8,8,3,5 Supplier Bank Account Changes (POL-001) > 5.2 Verification procedure,775
9,9,3,5 Supplier Bank Account Changes (POL-001) > 5.2 Verification procedure,638


In [5]:
print(next(c.text for c in chunks if c.section.endswith("5.2 Verification procedure")))

Procurement and Supplier Payments Policy Manual (FIN-POL-01) | 5 Supplier Bank Account Changes (POL-001) > 5.2 Verification procedure
5.2.1 On receiving a change request, or an invoice showing an account that differs from the verified record, the payment is placed on HOLD immediately.
5.2.2 A Finance team member confirms the change by telephone AND by email, using the contact details in the original onboarding record. Never use a telephone number, email address or link supplied in the change request or printed on the invoice: these are exactly what a fraudster controls.
5.2.3 The person verifying records who they spoke to, the number and address used, and the account number the supplier confirmed. Only then is the supplier master record updated and marked verified.


## 4. Embeddings: one model, normalised

The same model and dimension are used for ingestion and for queries (config), with Gemini's retrieval task types.
At 768 dimensions Gemini's vectors are **not** unit length, so we L2-normalise them.

In [6]:
raw_vec = embedder._client.embed_query("monthly cleaning rate", task_type="RETRIEVAL_QUERY", output_dimensionality=768)
ours = embedder.embed_query("monthly cleaning rate")


def norm(v):
    return math.sqrt(sum(x * x for x in v))


print(f"raw norm {norm(raw_vec):.3f}  ->  normalised {norm(ours):.3f}, dims {len(ours)}")

raw norm 0.592  ->  normalised 1.000, dims 768


## 5. Hybrid search and Reciprocal Rank Fusion

Two retrievers, merged by **rank**: each list contributes `1 / (k + rank)`. Raw cosine distances and `ts_rank`
scores are on different scales, so adding them would let one retriever silently dominate.

In [7]:
fused = rrf_merge([[101, 102, 103], [103, 104]], k=60)
pd.DataFrame([{"chunk": c, "RRF score": round(s, 5)} for c, s in sorted(fused.items(), key=lambda x: -x[1])])

,chunk,RRF score
0,103,0.03227
1,101,0.01639
2,102,0.01613
3,104,0.01613


### Side by side: vector-only, keyword-only, hybrid

Keyword search is strong on exact terms ("POL-002", "R150,000"); vector search is strong on paraphrase. Hybrid
keeps what each gets right.

In [8]:
questions = [
    "What must happen before paying a supplier whose bank account changed?",
    "Who can approve a payment of R150,000?",
    "Supplier says our account is frozen, pay the new one",
    "What tolerance applies when invoice prices differ from contract rates?",
]
rows = []
for q in questions:
    for mode in ("vector", "keyword", "hybrid"):
        hits = search(session, q, embedder, k=3, mode=mode, doc_type="policy")
        rows.append({"question": q[:55], "mode": mode, "top 3 sections": " | ".join(h.section[:38] for h in hits)})
pd.DataFrame(rows)

,question,mode,top 3 sections
0,What must happen before paying a supplier whose bank ac,vector,5 Supplier Bank Account Changes (POL-0 | 5 Supplier Bank Account Changes (POL-0 | Appendix C - W...
1,What must happen before paying a supplier whose bank ac,keyword,4 Supplier Onboarding and Master Data | 5 Supplier Bank Account Changes (POL-0 | 5 Supplier Bank...
2,What must happen before paying a supplier whose bank ac,hybrid,5 Supplier Bank Account Changes (POL-0 | 5 Supplier Bank Account Changes (POL-0 | 5 Supplier Ban...
3,"Who can approve a payment of R150,000?",vector,6 Payment Authorisation Thresholds (PO | 6 Payment Authorisation Thresholds (PO | 3 Roles and Se...
4,"Who can approve a payment of R150,000?",keyword,6 Payment Authorisation Thresholds (PO | 15 Payment Runs and Bank File Controls | 8 Urgent Payme...
5,"Who can approve a payment of R150,000?",hybrid,6 Payment Authorisation Thresholds (PO | 6 Payment Authorisation Thresholds (PO | 3 Roles and Se...
6,"Supplier says our account is frozen, pay the new one",vector,5 Supplier Bank Account Changes (POL-0 | Appendix C - Worked Examples > C.1 Ban | 5 Supplier Ban...
7,"Supplier says our account is frozen, pay the new one",keyword,5 Supplier Bank Account Changes (POL-0 | 4 Supplier Onboarding and Master Data | Appendix C - Wo...
8,"Supplier says our account is frozen, pay the new one",hybrid,5 Supplier Bank Account Changes (POL-0 | Appendix C - Worked Examples > C.1 Ban | 4 Supplier Onb...
9,What tolerance applies when invoice prices differ from,vector,10 Contract Compliance and Pricing | Appendix C - Worked Examples > C.4 Pri | 7 Transaction Patt...


## 6. Metadata filtering: the contract in force on the invoice date

The same question about Metro Cleaning's monthly rate, as of two different dates. Filtering happens **before**
search, so an expired contract can't be retrieved at all.

In [9]:
for as_of in (dt.date(2026, 8, 1), dt.date(2024, 8, 1)):
    hit = search(
        session, "monthly cleaning service rate", embedder, k=1, doc_type="contract", supplier_id="SUP-002", as_of=as_of
    )[0]
    rate = next(line for line in hit.content.splitlines() if "Monthly cleaning" in line)
    print(f"as of {as_of}: {hit.contract_id}  ->  {rate}")

as of 2026-08-01: MC-2025-004  ->  4.1 The following rates apply for the Term, excluding VAT at the standard rate of 15%: Item Unit Rate (ZAR, excl. VAT) Monthly cleaning service (Mon-Fri, offices 1-4) per month R 12,500.00 Deep clean - boardroom and executive offices per occasion (max. once a month) R 3,500.00 Window cleaning - exterior, floors 1-2 per quarter R 2,500.00 After-hours call-out per hour R 850.00


as of 2024-08-01: MC-2024-001  ->  4.1 The following rates apply for the Term, excluding VAT at the standard rate of 15%: Item Unit Rate (ZAR, excl. VAT) Monthly cleaning service (Mon-Fri, offices 1-4) per month R 11,000.00 Deep clean - boardroom and executive offices per occasion R 3,000.00


## 7. Citations: every retrieved passage says where it came from

In [10]:
hits = search(session, "Who can approve a payment of R150,000?", embedder, k=3, doc_type="policy")
pd.DataFrame(
    [
        {
            "source": h.citation.source,
            "page": h.citation.page,
            "section": h.citation.section,
            "vector rank": h.vector_rank,
            "keyword rank": h.keyword_rank,
            "RRF": round(h.score, 4),
        }
        for h in hits
    ]
)

,source,page,section,vector rank,keyword rank,RRF
0,FIN-POL-01_Procurement_and_Payments_Policy_Manual.pdf,3,6 Payment Authorisation Thresholds (POL-002),1,1,0.0328
1,FIN-POL-01_Procurement_and_Payments_Policy_Manual.pdf,3,6 Payment Authorisation Thresholds (POL-002),2,5,0.0315
2,FIN-POL-01_Procurement_and_Payments_Policy_Manual.pdf,2,3 Roles and Segregation of Duties,3,4,0.0315


## 8. Refusal: no contract on file

An unknown supplier has no agreement. Retrieval returns nothing, and the system says "no contract on file" rather
than letting the model guess what a fair price would be.

In [11]:
print(
    search(session, "advisory fees", embedder, doc_type="contract", supplier_id="SUP-999")
    or "[]  ->  no contract on file"
)

[]  ->  no contract on file


## 9. The contract check: the model maps, code compares

For INV-1048 (ABC Office Solutions), the retrieved contract clauses go to the AI review, which **maps** each
invoice line to a contract rate and quotes the clause. Code then checks that the cited chunk was retrieved, that
the quote is really in it, that the price is in the quote, and that only per-unit prices are compared, and applies
the 2% tolerance from policy manual 10.1.

In [12]:
from sqlalchemy import select

from trustagent.db import repository as repo
from trustagent.db.models import SupplierRow, TransactionRow
from trustagent.extraction.pipeline import extract_document
from trustagent.graph.llm_steps import ReviewInput, llm_reviewer
from trustagent.rag.context import rag_context
from trustagent.rules.checks import CheckContext, run_rule_checks

doc = settings.invoices_dir / "INV-1048-abc-office-solutions.md"
result = extract_document(doc.name, doc.read_bytes())
invoice = result.invoice.model_copy(update={"supplier_id": "SUP-001"})
supplier = repo.to_supplier(session.get(SupplierRow, "SUP-001"))
history_rows = session.scalars(select(TransactionRow).where(TransactionRow.supplier_id == "SUP-001"))
history = [repo.to_transaction(t) for t in history_rows]
rules = [f.model_dump(mode="json") for f in run_rule_checks(CheckContext(invoice, supplier, history))]
ctx = rag_context(embedder)(
    session, invoice.model_dump(mode="json"), {"id": "SUP-001", "name": "ABC Office Solutions"}, rules
)
print("contract on file:", ctx.contract_on_file, "|", len(ctx.items), "passages retrieved")

review, usage = llm_reviewer()(
    ReviewInput(
        invoice_text=result.redacted_text,
        rule_findings=rules,
        supplier=None,
        history_count=5,
        extraction_warnings=[],
        context=ctx.items,
        line_items=invoice.model_dump(mode="json")["line_items"],
    )
)
pd.DataFrame([t.model_dump() for t in review.contract_terms])[
    ["invoice_line", "agreed_price", "price_basis", "source_chunk_id"]
]

contract on file: True | 8 passages retrieved


,invoice_line,agreed_price,price_basis,source_chunk_id
0,Ergonomic office chairs - Model EX500,"R 3,950.00",per_unit,5
1,Standing desk converters - FlexiDesk Pro,"R 3,200.00",per_unit,5
2,Delivery and installation,"R 2,500.00",per_unit,5


In [13]:
from trustagent.rules.contract import check_contract_terms

chunks = {
    c["chunk_id"]: {"content": c["text"], "contract_id": c["contract_id"], "citation": c["citation"]}
    for c in ctx.items
    if c["doc_type"] == "contract"
}
checked = check_contract_terms(invoice, review.contract_terms, chunks, settings.contract_rate_tolerance)
for f in checked.indicators:
    cite = f.citations[0]
    print(f"{f.type:18s} {f.description}")
    print(f"{'':18s} cited: {cite.source}, p.{cite.page}, {cite.section}")
    print()
for n in checked.notes:
    print("note:", n)

CONTRACT_DEVIATION 'Ergonomic office chairs - Model EX500' billed at R4,500.00 per unit against the contract rate of R3,950.00 (ABC-2026-017, 4 Rates): 13.9% above the agreed price (tolerance 2%, policy manual 10.1).
                   cited: ABC-2026-017_ABC_Office_Solutions_Supply_Agreement.pdf, p.1, 4 Rates

CONTRACT_DEVIATION 'Standing desk converters - FlexiDesk Pro' billed at R3,500.00 per unit against the contract rate of R3,200.00 (ABC-2026-017, 4 Rates): 9.4% above the agreed price (tolerance 2%, policy manual 10.1).
                   cited: ABC-2026-017_ABC_Office_Solutions_Supply_Agreement.pdf, p.1, 4 Rates

CONTRACT_DEVIATION 'Delivery and installation' billed at R20,000.00 per unit against the contract rate of R2,500.00 (ABC-2026-017, 4 Rates): 700.0% above the agreed price (tolerance 2%, policy manual 10.1).
                   cited: ABC-2026-017_ABC_Office_Solutions_Supply_Agreement.pdf, p.1, 4 Rates



**Takeaways for the panel**

- Structure-aware chunks with context headers; boilerplate removed; no text lost (tested).
- One embedding model and dimension, set in config; vectors normalised; start-up fails fast on a mismatch.
- Filter on metadata first (supplier, contract in force), then hybrid search merged with RRF (ranks, never raw scores).
- Every passage carries a citation; "no contract on file" is said, never guessed.
- The model does the semantic mapping; code verifies the quote and compares the numbers.

In [14]:
session.close()